# GCD and EIS quantum-circuit figure notebook

This interactive notebook generates the four quantum-circuit schematics used to document the manuscript companion archive's quantum-assisted GCD and EIS workflows.

It includes:
- an 8-qubit, two-layer GCD discharge-window QAOA circuit;
- a 5-qubit, two-stage GCD hardware-efficient ansatz (HEA) circuit;
- a 7-qubit, three-layer EIS bounded-decoder HEA circuit;
- an EIS 21-qubit QUBO/QAOA circuit for 3-bit encoding of seven EIS parameters.

Scope and interpretation:
- The notebook is a circuit-generation and reproducibility helper for manuscript-facing schematic figures.
- Bounded decoding, objective/loss evaluation, and optimizer updates are classical steps and are intentionally kept outside the quantum gate sequence.
- The numerical angles and coefficients used here are representative deterministic settings for drawing and smoke checks. They are not a replacement for the committed GCD/EIS processed data products or the domain analysis scripts.
- Run the notebook cells from top to bottom to regenerate the four circuit figures interactively.


## Environment

Install the repository requirements before running this notebook:

```bash
python -m pip install -r requirements.txt
```

The notebook uses Qiskit, Qiskit Aer, Matplotlib, and pylatexenc. These dependencies are declared in the top-level `requirements.txt` so notebook reruns do not depend on executing a network-install cell.


## Circuit map

### 1. GCD discharge-window QAOA: 8 qubits, 2 layers
- Initial state: `|+>^n`, implemented with Hadamard gates on all qubits.
- Each QAOA layer contains linear `Rz(gamma_l a_i)` terms, nearest-neighbor `RZZ / U_len(gamma_l)` terms, and transverse-field `Rx(2 beta_l)` mixers.
- Measurement outcomes are decoded classically into a discharge-window mask.

### 2. GCD bounded HEA path: 5 qubits, 2 stages
- The circuit alternates shared `Ry(theta_l)` rotations, a `CZ` entangler chain, and shared `Ry(phi_l)` rotations.
- The builder supports both shared scalar layer parameters and per-qubit vectors.
- Bounded parameterization is treated as a classical preprocessing step before circuit construction.

### 3. EIS bounded-decoder HEA: 7 qubits, 3 layers
- The ansatz uses three `Ry(theta_{l,i})` rotation layers separated by brickwork entanglers.
- The seven decoded physical entries correspond to `[Rs, L0, R1, Q1, alpha1, Q2, alpha2]`.
- Quantum ansatz construction and physical-parameter decoding are kept as separate, auditable steps.

### 4. EIS QUBO/QAOA: 21 qubits, p=1
- The 21-qubit register represents 3 bits for each of the seven EIS parameters.
- The circuit contains a single QAOA cost/mixer layer and a classical bitstring decoder.
- The compact drawing keeps the full circuit logic while using a readable display for manuscript-scale figures.


In [ ]:
from __future__ import annotations

from collections.abc import Callable, Mapping, Sequence
from math import exp, pi
from typing import Any

import matplotlib.pyplot as plt
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import Parameter

try:
    from qiskit_aer import AerSimulator
except ImportError:  # pragma: no cover
    AerSimulator = None

EIS_PARAMETER_LABELS = ['Rs', 'L0', 'R1', 'Q1', 'alpha1', 'Q2', 'alpha2']


def _is_sequence_like(value: Any) -> bool:
    return isinstance(value, Sequence) and not isinstance(value, (str, bytes))


def _sigmoid(value: float) -> float:
    return 1.0 / (1.0 + exp(-float(value)))


def _expand_to_length(value: Any, length: int, name: str = 'value') -> list[float]:
    if _is_sequence_like(value):
        if len(value) != length:
            raise ValueError(f'{name} must have length {length}, got {len(value)}')
        return [float(item) for item in value]
    return [float(value)] * length


def _normalize_layers(layers: Sequence[Any], num_layers: int, num_qubits: int, name: str) -> list[list[float]]:
    if len(layers) != num_layers:
        raise ValueError(f'{name} must contain {num_layers} layers, got {len(layers)}')
    return [
        _expand_to_length(layer, num_qubits, f'{name}[{index}]')
        for index, layer in enumerate(layers)
    ]


def _decode_scalar(raw: float, lower: float, upper: float) -> float:
    return float(lower) + (float(upper) - float(lower)) * _sigmoid(raw)


def bounded_sigmoid_decode(raw: Any, p_min: Any, p_max: Any) -> Any:
    if isinstance(raw, Mapping):
        if isinstance(p_min, Mapping) and isinstance(p_max, Mapping):
            return {
                key: _decode_scalar(value, p_min[key], p_max[key])
                for key, value in raw.items()
            }
        lower = _expand_to_length(p_min, len(raw), 'p_min')
        upper = _expand_to_length(p_max, len(raw), 'p_max')
        return {
            key: _decode_scalar(value, lo, hi)
            for (key, value), lo, hi in zip(raw.items(), lower, upper)
        }

    if _is_sequence_like(raw):
        lower = _expand_to_length(p_min, len(raw), 'p_min')
        upper = _expand_to_length(p_max, len(raw), 'p_max')
        decoded = [_decode_scalar(value, lo, hi) for value, lo, hi in zip(raw, lower, upper)]
        return tuple(decoded) if isinstance(raw, tuple) else decoded

    return _decode_scalar(raw, p_min, p_max)


def nearest_neighbor_edges(num_qubits: int) -> list[tuple[int, int]]:
    return [(index, index + 1) for index in range(num_qubits - 1)]


def _normalize_weighted_edges(edges: Sequence[Any]) -> list[tuple[int, int, float]]:
    normalized = []
    for edge in edges:
        if len(edge) == 2:
            left, right = edge
            weight = 1.0
        elif len(edge) == 3:
            left, right, weight = edge
        else:
            raise ValueError('Each edge must be (i, j) or (i, j, weight)')
        normalized.append((int(left), int(right), float(weight)))
    return normalized


def _add_measurements(circuit: QuantumCircuit) -> None:
    for qubit in range(circuit.num_qubits):
        circuit.measure(qubit, qubit)


def _has_measurements(circuit: QuantumCircuit) -> bool:
    return any(instruction.operation.name == 'measure' for instruction in circuit.data)


def _with_terminal_measurements(circuit: QuantumCircuit, register_name: str = 'c') -> QuantumCircuit:
    working_circuit = circuit.copy()
    if _has_measurements(working_circuit):
        return working_circuit

    missing_clbits = working_circuit.num_qubits - working_circuit.num_clbits
    if missing_clbits > 0:
        working_circuit.add_register(ClassicalRegister(missing_clbits, register_name))

    for qubit in range(working_circuit.num_qubits):
        working_circuit.measure(qubit, working_circuit.clbits[qubit])
    return working_circuit


def _make_display_block(label: str, num_qubits: int):
    block = QuantumCircuit(num_qubits, name=label)
    return block.to_instruction(label=label)


def _apply_text_replacements(figure: Any, replacements: Mapping[str, str] | None = None) -> Any:
    if not figure.axes or not replacements:
        return figure

    axis = figure.axes[0]
    for text in axis.texts:
        label = text.get_text()
        if label.isdigit() and str(text.get_color()).upper() == '#FFFFFF':
            text.set_text('')
            continue
        if label in replacements:
            replacement = replacements[label]
            text.set_text(replacement)
            if '\\vdots' in replacement:
                text.set_fontsize(16)
                text.set_color('#666666')
            elif replacement.startswith('$'):
                text.set_fontsize(max(float(text.get_fontsize()), 10.0))
    return figure


def _style_multiqubit_display_blocks(
    figure: Any,
    block_marker: str,
    facecolor: str = '#F5EED8',
    edgecolor: str = '#5F7FA8',
    textcolor: str = '#1F2430',
    width: float = 1.5,
) -> Any:
    if not figure.axes:
        return figure

    axis = figure.axes[0]
    for text in axis.texts:
        label = text.get_text()
        if block_marker not in label:
            continue

        text_x, text_y = text.get_position()
        for patch in axis.patches:
            if type(patch).__name__ != 'Rectangle':
                continue
            try:
                patch_x = float(patch.get_x())
                patch_y = float(patch.get_y())
                patch_width = float(patch.get_width())
                patch_height = float(patch.get_height())
            except Exception:
                continue

            if patch_width < 0.7 or patch_height < 6.0:
                continue
            if not (patch_x <= text_x <= patch_x + patch_width and patch_y <= text_y <= patch_y + patch_height):
                continue

            center_x = patch_x + patch_width / 2.0
            center_y = patch_y + patch_height / 2.0
            patch.set_x(center_x - width / 2.0)
            patch.set_width(width)
            patch.set_facecolor(facecolor)
            patch.set_edgecolor(edgecolor)
            patch.set_linewidth(2.0)
            text.set_color(textcolor)
            text.set_fontsize(max(float(text.get_fontsize()), 14.0))
            text.set_horizontalalignment('center')
            text.set_verticalalignment('center')
            text.set_position((center_x, center_y))
            break
    return figure


def _shift_artists_horizontally(figure: Any, x_threshold: float, delta: float) -> Any:
    if not figure.axes or abs(float(delta)) < 1e-12:
        return figure

    axis = figure.axes[0]
    threshold = float(x_threshold)
    shift = float(delta)

    for patch in axis.patches:
        moved = False
        if hasattr(patch, 'get_x') and hasattr(patch, 'set_x'):
            try:
                patch_x = float(patch.get_x())
            except Exception:
                patch_x = None
            if patch_x is not None and patch_x >= threshold:
                patch.set_x(patch_x + shift)
                moved = True
        if moved:
            continue

        center = getattr(patch, 'center', None)
        if center is not None:
            try:
                center_x, center_y = center
                center_x = float(center_x)
                center_y = float(center_y)
            except Exception:
                center_x = center_y = None
            if center_x is not None and center_x >= threshold:
                patch.center = (center_x + shift, center_y)
                continue

        if hasattr(patch, 'get_xy') and hasattr(patch, 'set_xy'):
            try:
                vertices = patch.get_xy()
                shifted_vertices = [(float(x) + shift, float(y)) for x, y in vertices]
            except Exception:
                continue
            if shifted_vertices and min(x for x, _ in shifted_vertices) - shift >= threshold:
                patch.set_xy(shifted_vertices)

    for text in axis.texts:
        try:
            text_x, text_y = text.get_position()
            text_x = float(text_x)
            text_y = float(text_y)
        except Exception:
            continue
        if text_x >= threshold:
            text.set_position((text_x + shift, text_y))

    for line in axis.lines:
        x_values = [float(value) for value in line.get_xdata()]
        if not x_values or min(x_values) < threshold:
            continue
        line.set_xdata([value + shift for value in x_values])

    x_min, x_max = axis.get_xlim()
    axis.set_xlim(x_min, x_max + shift)
    return figure


def _expand_first_discharge_gap(figure: Any, delta: float = 0.35) -> Any:
    if not figure.axes:
        return figure

    axis = figure.axes[0]
    rz_centers = [
        float(text.get_position()[0])
        for text in axis.texts
        if 'R_Z' in text.get_text()
    ]
    if not rz_centers:
        return figure

    first_rz_left = min(rz_centers) - 0.325
    return _shift_artists_horizontally(figure, x_threshold=first_rz_left, delta=delta)


def figure_text_labels(figure: Any) -> list[str]:
    if not figure.axes:
        return []
    return [text.get_text() for text in figure.axes[0].texts]


def draw_symbolic_circuit(
    circuit: QuantumCircuit,
    text_replacements: Mapping[str, str] | None = None,
    fold: int = -1,
    add_measurements_if_missing: bool = True,
    show: bool = True,
):
    working_circuit = _with_terminal_measurements(circuit) if add_measurements_if_missing else circuit.copy()
    figure = working_circuit.draw('mpl', fold=fold)
    _apply_text_replacements(figure, text_replacements)
    if show:
        plt.show()
    return figure


def measure_all_and_sample(
    circuit: QuantumCircuit,
    shots: int = 1024,
    backend: Any = None,
    seed_simulator: int = 1234,
) -> dict[str, int]:
    working_circuit = _with_terminal_measurements(circuit)

    if backend is None:
        if AerSimulator is None:
            raise ImportError('qiskit-aer is required for sampling in this notebook')
        backend = AerSimulator(seed_simulator=seed_simulator)

    compiled = transpile(working_circuit, backend)
    result = backend.run(compiled, shots=shots).result()
    return result.get_counts()


def best_shot(counts: Mapping[str, int]) -> str:
    return max(counts.items(), key=lambda item: (item[1], item[0]))[0]


def decode_mask_from_bitstring(
    bitstring: str,
    rule: str | Callable[[list[int]], Any] = 'binary',
    reverse_bits: bool = True,
) -> Any:
    clean = bitstring.replace(' ', '')
    ordered = clean[::-1] if reverse_bits else clean
    mask = [int(bit) for bit in ordered]

    if callable(rule):
        return rule(mask)
    if rule == 'binary':
        return {
            'bitstring': clean,
            'mask': mask,
            'active_indices': [index for index, bit in enumerate(mask) if bit],
        }
    if rule == 'window_indices':
        return {
            'bitstring': clean,
            'mask': mask,
            'windows': [index for index, bit in enumerate(mask) if bit],
        }
    raise ValueError(f'Unsupported decode rule: {rule}')


def decode_eis_bits(
    bitstring: str,
    group_size: int = 3,
    eta: float = 0.08,
    labels: Sequence[str] = EIS_PARAMETER_LABELS,
    reverse_bits: bool = True,
) -> dict[str, float]:
    clean = bitstring.replace(' ', '')
    expected_bits = group_size * len(labels)
    if len(clean) != expected_bits:
        raise ValueError(f'Expected {expected_bits} bits, got {len(clean)}')
    ordered = clean[::-1] if reverse_bits else clean
    chunks = [ordered[index:index + group_size] for index in range(0, len(ordered), group_size)]
    values = [eta * int(chunk, 2) for chunk in chunks]
    return dict(zip(labels, values))


def named_parameter_dict(values: Sequence[float], labels: Sequence[str] = EIS_PARAMETER_LABELS) -> dict[str, float]:
    if len(values) != len(labels):
        raise ValueError(f'Expected {len(labels)} values, got {len(values)}')
    return dict(zip(labels, [float(value) for value in values]))


def _apply_chain_entangler(circuit: QuantumCircuit, entangler: str, num_qubits: int) -> None:
    for left in range(num_qubits - 1):
        if entangler == 'cz_chain':
            circuit.cz(left, left + 1)
        elif entangler == 'cx_chain':
            circuit.cx(left, left + 1)
        else:
            raise ValueError(f'Unsupported entangler: {entangler}')


def _apply_brickwork_entangler(
    circuit: QuantumCircuit,
    entangler_pattern: str,
    offset: int,
    num_qubits: int,
) -> None:
    for left in range(offset, num_qubits - 1, 2):
        if entangler_pattern == 'brickwork_cnot':
            circuit.cx(left, left + 1)
        elif entangler_pattern == 'brickwork_cz':
            circuit.cz(left, left + 1)
        else:
            raise ValueError(f'Unsupported entangler pattern: {entangler_pattern}')


def build_discharge_window_qaoa(
    num_qubits: int = 8,
    gammas: Sequence[float] = (0.55, 0.30),
    betas: Sequence[float] = (0.22, 0.15),
    a: Sequence[float] | None = None,
    zz_edges: Sequence[Any] | None = None,
    add_measurements: bool = False,
) -> QuantumCircuit:
    if len(gammas) != len(betas):
        raise ValueError('gammas and betas must have the same length')
    a_values = _expand_to_length(a if a is not None else [1.0] * num_qubits, num_qubits, 'a')
    weighted_edges = _normalize_weighted_edges(zz_edges or nearest_neighbor_edges(num_qubits))
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='discharge_window_qaoa')

    for qubit in range(num_qubits):
        circuit.h(qubit)

    for layer_index, (gamma, beta) in enumerate(zip(gammas, betas), start=1):
        for qubit, coefficient in enumerate(a_values):
            circuit.rz(2.0 * float(gamma) * coefficient, qubit)
        for left, right, weight in weighted_edges:
            circuit.rzz(2.0 * float(gamma) * weight, left, right)
        for qubit in range(num_qubits):
            circuit.rx(2.0 * float(beta), qubit)
        if layer_index != len(gammas):
            circuit.barrier()

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def build_gcd_hea(
    num_qubits: int = 5,
    theta_layers: Sequence[Any] = (0.45, 0.20),
    phi_layers: Sequence[Any] = (1.00, 0.35),
    entangler: str = 'cz_chain',
    add_measurements: bool = False,
) -> QuantumCircuit:
    theta = _normalize_layers(theta_layers, 2, num_qubits, 'theta_layers')
    phi = _normalize_layers(phi_layers, 2, num_qubits, 'phi_layers')
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='gcd_hea')

    for layer_index in range(2):
        for qubit, angle in enumerate(theta[layer_index]):
            circuit.ry(angle, qubit)
        _apply_chain_entangler(circuit, entangler, num_qubits)
        for qubit, angle in enumerate(phi[layer_index]):
            circuit.ry(angle, qubit)
        if layer_index == 0:
            circuit.barrier()

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def build_eis_decoder_hea(
    num_qubits: int = 7,
    theta_layers: Sequence[Any] | None = None,
    entangler_pattern: str = 'brickwork_cnot',
    add_measurements: bool = False,
) -> QuantumCircuit:
    if theta_layers is None:
        theta_layers = [
            [0.08 * (qubit + 1) for qubit in range(num_qubits)],
            [0.06 * (qubit + 1) for qubit in range(num_qubits)],
            [0.04 * (qubit + 1) for qubit in range(num_qubits)],
        ]

    theta = _normalize_layers(theta_layers, 3, num_qubits, 'theta_layers')
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='eis_decoder_hea')

    for qubit, angle in enumerate(theta[0]):
        circuit.ry(angle, qubit)
    _apply_brickwork_entangler(circuit, entangler_pattern, offset=0, num_qubits=num_qubits)
    circuit.barrier()

    for qubit, angle in enumerate(theta[1]):
        circuit.ry(angle, qubit)
    _apply_brickwork_entangler(circuit, entangler_pattern, offset=1, num_qubits=num_qubits)
    circuit.barrier()

    for qubit, angle in enumerate(theta[2]):
        circuit.ry(angle, qubit)

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def build_eis_qubo_qaoa(
    num_qubits: int = 21,
    gamma: float = 0.55,
    beta: float = 0.25,
    h: Sequence[float] | None = None,
    couplings: Sequence[Any] | None = None,
    add_measurements: bool = False,
) -> QuantumCircuit:
    h_values = _expand_to_length(h if h is not None else [0.15] * num_qubits, num_qubits, 'h')
    weighted_edges = _normalize_weighted_edges(couplings or nearest_neighbor_edges(num_qubits))
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='eis_qubo_qaoa')

    for qubit in range(num_qubits):
        circuit.h(qubit)
    for qubit, coefficient in enumerate(h_values):
        circuit.rz(2.0 * float(gamma) * coefficient, qubit)
    for left, right, weight in weighted_edges:
        circuit.rzz(2.0 * float(gamma) * weight, left, right)
    for qubit in range(num_qubits):
        circuit.rx(2.0 * float(beta), qubit)

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def build_discharge_window_qaoa_display(
    num_qubits: int = 8,
    zz_edges: Sequence[Any] | None = None,
    add_measurements: bool = False,
) -> QuantumCircuit:
    weighted_edges = _normalize_weighted_edges(zz_edges or nearest_neighbor_edges(num_qubits))
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='discharge_window_qaoa_display')
    rz_display_params = [Parameter('G1AI'), Parameter('G2AI')]
    zz_display_params = [Parameter('G1ZZ'), Parameter('G2ZZ')]
    block_labels = ['ULEN1', 'ULEN2']
    rx_display_params = [Parameter('TWOB1'), Parameter('TWOB2')]

    for qubit in range(num_qubits):
        circuit.h(qubit)

    for layer_index in range(2):
        for qubit in range(num_qubits):
            circuit.rz(rz_display_params[layer_index], qubit)
        for left, right, _ in weighted_edges:
            circuit.rzz(zz_display_params[layer_index], left, right)
        circuit.append(_make_display_block(block_labels[layer_index], num_qubits), range(num_qubits))
        for qubit in range(num_qubits):
            circuit.rx(rx_display_params[layer_index], qubit)
        if layer_index == 0:
            circuit.barrier()

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def _discharge_window_display_replacements() -> dict[str, str]:
    return {
        'G1AI': r'$\gamma_{\,1}\, a_i$',
        'G2AI': r'$\gamma_{\,2}\, a_i$',
        '$\\mathrm{ZZ}$ (G1ZZ)': r'$\mathrm{ZZ}\,(\gamma_{\,1})$',
        '$\\mathrm{ZZ}$ (G2ZZ)': r'$\mathrm{ZZ}\,(\gamma_{\,2})$',
        'ULEN1': r'$U_{\mathrm{len}}(\gamma_{\,1})$',
        'ULEN2': r'$U_{\mathrm{len}}(\gamma_{\,2})$',
        'TWOB1': r'$2\,\beta_{1}$',
        'TWOB2': r'$2\,\beta_{2}$',
    }


def draw_discharge_window_qaoa_display(fold: int = -1, show: bool = True):
    figure = draw_symbolic_circuit(
        build_discharge_window_qaoa_display(add_measurements=False),
        text_replacements=_discharge_window_display_replacements(),
        fold=fold,
        add_measurements_if_missing=True,
        show=show,
    )
    return _style_multiqubit_display_blocks(figure, block_marker='U_{\\mathrm{len}}')


def build_gcd_hea_display(
    num_qubits: int = 5,
    entangler: str = 'cz_chain',
    add_measurements: bool = False,
) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='gcd_hea_display')
    layer_display_params = [
        Parameter('THETA1'),
        Parameter('PHI1'),
        Parameter('THETA2'),
        Parameter('PHI2'),
    ]

    for qubit in range(num_qubits):
        circuit.ry(layer_display_params[0], qubit)
    _apply_chain_entangler(circuit, entangler, num_qubits)
    for qubit in range(num_qubits):
        circuit.ry(layer_display_params[1], qubit)
    circuit.barrier()
    for qubit in range(num_qubits):
        circuit.ry(layer_display_params[2], qubit)
    _apply_chain_entangler(circuit, entangler, num_qubits)
    for qubit in range(num_qubits):
        circuit.ry(layer_display_params[3], qubit)

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def _gcd_hea_display_replacements() -> dict[str, str]:
    return {
        'THETA1': r'$\theta_{1}$',
        'PHI1': r'$\phi_{1}$',
        'THETA2': r'$\theta_{2}$',
        'PHI2': r'$\phi_{2}$',
    }


def draw_gcd_hea_display(fold: int = -1, show: bool = True):
    return draw_symbolic_circuit(
        build_gcd_hea_display(add_measurements=False),
        text_replacements=_gcd_hea_display_replacements(),
        fold=fold,
        add_measurements_if_missing=True,
        show=show,
    )


def build_eis_decoder_hea_display(
    num_qubits: int = 7,
    entangler_pattern: str = 'brickwork_cnot',
    add_measurements: bool = False,
) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits, num_qubits if add_measurements else 0, name='eis_decoder_hea_display')
    decoder_display_params = [Parameter('THETA1I'), Parameter('THETA2I'), Parameter('THETA3I')]

    for qubit in range(num_qubits):
        circuit.ry(decoder_display_params[0], qubit)
    _apply_brickwork_entangler(circuit, entangler_pattern, offset=0, num_qubits=num_qubits)
    circuit.barrier()
    for qubit in range(num_qubits):
        circuit.ry(decoder_display_params[1], qubit)
    _apply_brickwork_entangler(circuit, entangler_pattern, offset=1, num_qubits=num_qubits)
    circuit.barrier()
    for qubit in range(num_qubits):
        circuit.ry(decoder_display_params[2], qubit)

    if add_measurements:
        _add_measurements(circuit)
    return circuit


def _eis_decoder_hea_display_replacements() -> dict[str, str]:
    return {
        'THETA1I': r'$\theta_{1,i}$',
        'THETA2I': r'$\theta_{2,i}$',
        'THETA3I': r'$\theta_{3,i}$',
    }


def draw_eis_decoder_hea_display(fold: int = -1, show: bool = True):
    return draw_symbolic_circuit(
        build_eis_decoder_hea_display(add_measurements=False),
        text_replacements=_eis_decoder_hea_display_replacements(),
        fold=fold,
        add_measurements_if_missing=True,
        show=show,
    )


def build_eis_qubo_qaoa_compact_display(
    num_qubits: int = 21,
    gamma: float = 0.55,
    beta: float = 0.25,
    h: Sequence[float] | None = None,
    couplings: Sequence[Any] | None = None,
    visible_qubits: Sequence[int] = (0, 1, 2, 3, 4, 5, 6, 7, 19, 20),
    add_measurements: bool = True,
) -> QuantumCircuit:
    visible = [int(qubit) for qubit in visible_qubits]
    if visible != sorted(visible):
        raise ValueError('visible_qubits must be sorted in ascending order')
    if len(set(visible)) != len(visible):
        raise ValueError('visible_qubits must not contain duplicates')
    if any(qubit < 0 or qubit >= num_qubits for qubit in visible):
        raise ValueError('visible_qubits must stay within the full circuit range')

    registers: list[QuantumRegister] = []
    qubit_map: dict[int, Any] = {}
    inserted_gap = False
    for index, qubit in enumerate(visible):
        if index > 0 and qubit - visible[index - 1] > 1 and not inserted_gap:
            registers.append(QuantumRegister(1, 'qgap'))
            inserted_gap = True
        qreg = QuantumRegister(1, f'q{qubit}')
        registers.append(qreg)
        qubit_map[qubit] = qreg[0]

    if add_measurements:
        creg = ClassicalRegister(len(visible), 'c')
        circuit = QuantumCircuit(*registers, creg, name='eis_qubo_qaoa_compact')
    else:
        creg = None
        circuit = QuantumCircuit(*registers, name='eis_qubo_qaoa_compact')

    rz_display_param = Parameter('TWOGHI')
    zz_display_param = Parameter('TWOGJ')
    rx_display_param = Parameter('TWOB')

    for qubit in visible:
        circuit.h(qubit_map[qubit])
    for qubit in visible:
        circuit.rz(rz_display_param, qubit_map[qubit])
    for left, right in nearest_neighbor_edges(num_qubits):
        if left in qubit_map and right in qubit_map:
            circuit.rzz(zz_display_param, qubit_map[left], qubit_map[right])
    for qubit in visible:
        circuit.rx(rx_display_param, qubit_map[qubit])

    if add_measurements and creg is not None:
        for classical_index, qubit in enumerate(visible):
            circuit.measure(qubit_map[qubit], creg[classical_index])
    return circuit


def _eis_qubo_qaoa_compact_display_replacements() -> dict[str, str]:
    return {
        'TWOGHI': r'$2\,\gamma\, h_i$',
        '$\\mathrm{ZZ}$ (TWOGJ)': r'$\mathrm{ZZ}\,(2\,\gamma\, J_{i,i+1})$',
        'TWOB': r'$2\,\beta$',
        '${qgap}$': r'$\vdots$',
    }


def draw_eis_qubo_qaoa_compact_view(
    num_qubits: int = 21,
    gamma: float = 0.55,
    beta: float = 0.25,
    h: Sequence[float] | None = None,
    couplings: Sequence[Any] | None = None,
    visible_qubits: Sequence[int] = (0, 1, 2, 3, 4, 5, 6, 7, 19, 20),
    add_measurements: bool = True,
    fold: int = 60,
    show: bool = True,
):
    compact_circuit = build_eis_qubo_qaoa_compact_display(
        num_qubits=num_qubits,
        gamma=gamma,
        beta=beta,
        h=h,
        couplings=couplings,
        visible_qubits=visible_qubits,
        add_measurements=add_measurements,
    )
    figure = draw_symbolic_circuit(
        compact_circuit,
        text_replacements=_eis_qubo_qaoa_compact_display_replacements(),
        fold=fold,
        add_measurements_if_missing=False,
        show=show,
    )
    return figure


def discharge_window_terms(
    mask: Mapping[str, Any] | Sequence[int],
    a: Sequence[float] | None = None,
    lambda_penalty: float = 1.0,
    mu_penalty: float = 1.0,
    target_active: int | None = None,
) -> dict[str, float]:
    mask_values = list(mask['mask']) if isinstance(mask, Mapping) else [int(bit) for bit in mask]
    if a is None:
        a = [1.0] * len(mask_values)
    weights = _expand_to_length(a, len(mask_values), 'a')
    if target_active is None:
        target_active = len(mask_values) // 2

    linear_term = float(
        lambda_penalty * sum((mask_values[index] - mask_values[index + 1]) ** 2 for index in range(len(mask_values) - 1))
    )
    zz_term = float(mu_penalty * (sum(mask_values) - int(target_active)) ** 2)
    mixer_term = float(sum(weight * bit for weight, bit in zip(weights, mask_values)))
    total = float(linear_term + zz_term - mixer_term)
    return {
        'linear_term': linear_term,
        'zz_term': zz_term,
        'mixer_term': mixer_term,
        'target_active': float(target_active),
        'total': total,
    }


def objective_discharge_window(
    mask: Mapping[str, Any] | Sequence[int],
    a: Sequence[float] | None = None,
    lambda_penalty: float = 1.0,
    mu_penalty: float = 1.0,
    target_active: int | None = None,
) -> float:
    return discharge_window_terms(
        mask,
        a=a,
        lambda_penalty=lambda_penalty,
        mu_penalty=mu_penalty,
        target_active=target_active,
    )['total']


def decode_gcd_parameterization(
    raw_u: Sequence[float],
    lower: float = 0.0,
    upper: float = pi,
    labels: Sequence[str] = ('theta_1', 'theta_2', 'phi_1', 'phi_2'),
) -> dict[str, float]:
    decoded = bounded_sigmoid_decode(list(raw_u), lower, upper)
    return dict(zip(labels, decoded))


def objective_quantum_path(
    decoded_params: Mapping[str, float] | Sequence[float],
    target: Sequence[float] | None = None,
    mask: Mapping[str, Any] | Sequence[int] | None = None,
    scs_weight: float = 1.0,
    gcd_weight: float = 0.2,
) -> dict[str, float]:
    values = list(decoded_params.values()) if isinstance(decoded_params, Mapping) else [float(value) for value in decoded_params]
    target_values = _expand_to_length(target if target is not None else [1.0] * len(values), len(values), 'target')
    scs_like = float(sum((value - target_value) ** 2 for value, target_value in zip(values, target_values)))
    gcd_like = 0.0
    if mask is not None:
        mask_values = list(mask['mask']) if isinstance(mask, Mapping) else [int(bit) for bit in mask]
        gcd_like = float(sum(mask_values) / max(len(mask_values), 1))
    total = float(scs_weight * scs_like + gcd_weight * gcd_like)
    return {'scs_like': scs_like, 'gcd_like': gcd_like, 'total': total}


def objective_gcd(
    decoded_params: Mapping[str, float] | Sequence[float],
    target: Sequence[float] | None = None,
    mask: Mapping[str, Any] | Sequence[int] | None = None,
) -> float:
    return objective_quantum_path(decoded_params, target=target, mask=mask)['total']


def objective_eis_surrogate(
    decoded_params: Mapping[str, float] | Sequence[float],
    omega: Sequence[float],
    target_curve: Sequence[complex] | None = None,
) -> dict[str, Any]:
    values = list(decoded_params.values()) if isinstance(decoded_params, Mapping) else [float(value) for value in decoded_params]
    omega_values = [float(value) for value in omega]
    scale = sum(values) / max(len(values), 1)
    prediction = [complex(scale / (1.0 + frequency), -scale * frequency / (1.0 + frequency ** 2)) for frequency in omega_values]
    if target_curve is None:
        target_curve = [0.0j] * len(omega_values)
    loss = sum(abs(pred - target) ** 2 for pred, target in zip(prediction, target_curve)) / max(len(omega_values), 1)
    return {'loss': float(loss), 'prediction': prediction}


def run_cobyla_like_loop(
    initial_u: Sequence[float],
    objective: Callable[[Any], float],
    decoder: Callable[[Sequence[float]], Any] | None = None,
    steps: int = 5,
    step_size: float = 0.15,
    bounds: tuple[float, float] = (0.0, 1.0),
) -> dict[str, Any]:
    decoder = decoder or (lambda value: value)
    lower, upper = bounds
    best_u = [float(value) for value in initial_u]
    best_decoded = decoder(best_u)
    best_value = float(objective(best_decoded))
    history = [{'step': 0, 'u': list(best_u), 'objective': best_value}]

    current_step_size = float(step_size)
    for step in range(1, steps + 1):
        improved = False
        for index in range(len(best_u)):
            for delta in (-current_step_size, current_step_size):
                candidate = list(best_u)
                candidate[index] = min(upper, max(lower, candidate[index] + delta))
                decoded_candidate = decoder(candidate)
                candidate_value = float(objective(decoded_candidate))
                history.append({'step': step, 'u': list(candidate), 'objective': candidate_value})
                if candidate_value < best_value:
                    best_u = candidate
                    best_decoded = decoded_candidate
                    best_value = candidate_value
                    improved = True
        if not improved:
            current_step_size *= 0.5

    return {
        'best_u': best_u,
        'best_value': best_value,
        'decoded': best_decoded,
        'history': history,
    }


## GCD circuit 1: discharge-window QAOA

This section builds the 8-qubit, two-layer QAOA circuit used to represent the GCD stable-window selection step.

The helper functions separate three QUBO-like components for transparency:
- local preference term: `sum_i a_i x_i`
- smoothness/window-contiguity term: `lambda sum_i (x_i - x_{i+1})^2`
- cardinality penalty: `mu (sum_i x_i - m)^2`

These terms are evaluated classically after sampling. The quantum circuit itself contains only the QAOA state preparation, cost rotations, mixer rotations, and terminal measurement.


In [ ]:
discharge_a = [0.90, 1.05, 0.85, 1.10, 0.95, 0.75, 1.00, 0.88]
discharge_qaoa = build_discharge_window_qaoa(
    gammas=[0.52, 0.31],
    betas=[0.20, 0.14],
    a=discharge_a,
    add_measurements=False,
)

discharge_counts = measure_all_and_sample(discharge_qaoa, shots=512)
discharge_best = best_shot(discharge_counts)
discharge_mask = decode_mask_from_bitstring(discharge_best)
discharge_terms = discharge_window_terms(
    discharge_mask,
    a=discharge_a,
    lambda_penalty=1.30,
    mu_penalty=0.80,
    target_active=4,
)
discharge_loss = objective_discharge_window(
    discharge_mask,
    a=discharge_a,
    lambda_penalty=1.30,
    mu_penalty=0.80,
    target_active=4,
)

print('best shot:', discharge_best)
print('decoded mask:', discharge_mask)
print('compiled QUBO-like terms:', {key: round(value, 4) for key, value in discharge_terms.items()})
print('representative discharge objective:', round(discharge_loss, 4))
draw_discharge_window_qaoa_display(fold=-1)


## GCD circuit 2: bounded HEA path

This section builds the 5-qubit GCD hardware-efficient ansatz used to illustrate the bounded quantum-path parameterization.

The workflow is intentionally split into:
- `decode_gcd_parameterization(...)`: maps raw optimizer variables into bounded circuit angles;
- `build_gcd_hea(...)`: constructs the quantum circuit from the decoded angles;
- `objective_quantum_path(...)` / `objective_gcd(...)`: evaluates the classical score used for the demonstration.

This keeps the manuscript schematic faithful to the hybrid workflow: the quantum circuit prepares and measures trial states, while parameter bounds and objective evaluation remain classical.


In [ ]:
raw_gcd_u = [-1.10, -0.20, 0.35, 0.95]
decoded_gcd_named = decode_gcd_parameterization(raw_gcd_u)
decoded_gcd_angles = list(decoded_gcd_named.values())
gcd_theta_layers = [decoded_gcd_named['theta_1'], decoded_gcd_named['theta_2']]
gcd_phi_layers = [decoded_gcd_named['phi_1'], decoded_gcd_named['phi_2']]

gcd_hea = build_gcd_hea(
    theta_layers=gcd_theta_layers,
    phi_layers=gcd_phi_layers,
    entangler='cz_chain',
    add_measurements=False,
)

gcd_counts = measure_all_and_sample(gcd_hea, shots=512)
gcd_best = best_shot(gcd_counts)
gcd_bit_mask = decode_mask_from_bitstring(gcd_best)
gcd_terms = objective_quantum_path(
    decoded_gcd_named,
    target=[0.60, 1.10, 1.80, 2.20],
    mask=gcd_bit_mask,
    scs_weight=1.0,
    gcd_weight=0.25,
)
gcd_loss = objective_gcd(decoded_gcd_named, target=[0.60, 1.10, 1.80, 2.20], mask=gcd_bit_mask)
gcd_optimizer_demo = run_cobyla_like_loop(
    initial_u=[0.25, 0.50, 0.75, 0.10],
    decoder=decode_gcd_parameterization,
    objective=lambda params: objective_gcd(params, target=[0.80, 1.30, 1.60, 2.10]),
    steps=2,
)

print('decoded shared angles:', {key: round(value, 4) for key, value in decoded_gcd_named.items()})
print('best shot:', gcd_best)
print('decoded bit mask:', gcd_bit_mask)
print('quantum-path objective terms:', {key: round(value, 4) for key, value in gcd_terms.items()})
print('representative GCD objective:', round(gcd_loss, 4))
print('COBYLA-like demo best value:', round(gcd_optimizer_demo['best_value'], 4))
draw_gcd_hea_display(fold=-1)


## EIS circuit 1: bounded-decoder HEA

This section builds the 7-qubit, three-layer EIS decoder HEA.

The example demonstrates:
- brickwork entanglement across seven EIS parameter channels;
- bounded decoding from raw optimizer variables to physical EIS parameters;
- a lightweight surrogate score used only to exercise the circuit-generation workflow.

The decoded physical labels match the EIS manuscript branch: `Rs`, `L0`, `R1`, `Q1`, `alpha1`, `Q2`, and `alpha2`.


In [ ]:
eis_raw_physical = {
    'Rs': -0.70,
    'L0': -0.25,
    'R1': 0.40,
    'Q1': 0.10,
    'alpha1': 0.55,
    'Q2': -0.35,
    'alpha2': 0.85,
}

eis_physical_min = {
    'Rs': 0.01,
    'L0': 1e-6,
    'R1': 0.05,
    'Q1': 1e-4,
    'alpha1': 0.20,
    'Q2': 1e-4,
    'alpha2': 0.20,
}

eis_physical_max = {
    'Rs': 10.0,
    'L0': 1.0,
    'R1': 25.0,
    'Q1': 2.0,
    'alpha1': 1.00,
    'Q2': 2.0,
    'alpha2': 1.00,
}

decoded_eis_params = bounded_sigmoid_decode(eis_raw_physical, eis_physical_min, eis_physical_max)
theta_layers_eis = [
    [0.12 * (qubit + 1) for qubit in range(7)],
    [0.08 * (qubit + 1) for qubit in range(7)],
    [0.05 * (qubit + 1) for qubit in range(7)],
]

eis_decoder_circuit = build_eis_decoder_hea(theta_layers=theta_layers_eis, add_measurements=False)
eis_decoder_counts = measure_all_and_sample(eis_decoder_circuit, shots=512)
eis_decoder_best = best_shot(eis_decoder_counts)
eis_decoder_mask = decode_mask_from_bitstring(eis_decoder_best)
eis_surrogate_demo = objective_eis_surrogate(decoded_eis_params, omega=[0.1, 1.0, 10.0, 100.0])

print('decoded physical parameters:')
for key, value in decoded_eis_params.items():
    print(f'  {key}: {value:.6f}')
print('best shot:', eis_decoder_best)
print('decoded bit mask:', eis_decoder_mask)
print('representative EIS surrogate loss:', round(eis_surrogate_demo['loss'], 6))
draw_eis_decoder_hea_display(fold=-1)


## EIS circuit 2: EIS 21-qubit QUBO/QAOA circuit

This section builds the 21-qubit EIS QUBO/QAOA circuit, where each of the seven EIS parameters is represented by three binary variables.

The sampled bitstring is decoded as a 3-bit x 7-parameter vector and then evaluated with the same lightweight surrogate helper used above. The compact view keeps the first and last register blocks visible while indicating the omitted middle wires, making the diagram suitable for manuscript-scale circuit schematics.


In [ ]:
qubo_h = [0.05 + 0.01 * (index % 7) for index in range(21)]
qubo_couplings = [(index, index + 1, 0.40 + 0.05 * (index % 3)) for index in range(20)]

eis_qubo_qaoa = build_eis_qubo_qaoa(
    gamma=0.48,
    beta=0.21,
    h=qubo_h,
    couplings=qubo_couplings,
    add_measurements=False,
)

eis_qubo_counts = measure_all_and_sample(eis_qubo_qaoa, shots=256)
eis_qubo_best = best_shot(eis_qubo_counts)
decoded_qubo_params = decode_eis_bits(eis_qubo_best, eta=0.08)
eis_qubo_surrogate = objective_eis_surrogate(decoded_qubo_params, omega=[0.2, 2.0, 20.0, 200.0])

print('best shot:', eis_qubo_best)
print('decoded 3-bit x 7 parameters:')
for key, value in decoded_qubo_params.items():
    print(f'  {key}: {value:.4f}')
print('representative QUBO surrogate loss:', round(eis_qubo_surrogate['loss'], 6))
print('compact view: q0-q7, ..., q19-q20')
draw_eis_qubo_qaoa_compact_view(gamma=0.48, beta=0.21, h=qubo_h, couplings=qubo_couplings, add_measurements=True)


## Notebook smoke checks

The checks below verify that the builders still produce the expected gate structure, qubit counts, display labels, bounded decoders, and helper outputs. They are intentionally lightweight so the notebook remains interactive and easy to rerun during figure preparation.


In [ ]:
decoded_test = bounded_sigmoid_decode([0.0, -2.0, 2.0], p_min=-1.0, p_max=1.0)
assert all(-1.0 <= value <= 1.0 for value in decoded_test)

discharge_term_test = discharge_window_terms([1, 0, 1, 1], a=[0.5, 0.5, 0.5, 0.5], target_active=2)
assert set(discharge_term_test.keys()) == {'linear_term', 'zz_term', 'mixer_term', 'target_active', 'total'}

gcd_parameter_test = decode_gcd_parameterization([0.0, 0.1, 0.2, 0.3])
assert list(gcd_parameter_test.keys()) == ['theta_1', 'theta_2', 'phi_1', 'phi_2']

decoded_bits = decode_eis_bits('0' * 21)
assert len(decoded_bits) == 7
assert set(decoded_bits.keys()) == set(EIS_PARAMETER_LABELS)

discharge_ops = build_discharge_window_qaoa(add_measurements=False).count_ops()
assert discharge_ops.get('h', 0) == 8
assert discharge_ops.get('rz', 0) == 16
assert discharge_ops.get('rzz', 0) == 14
assert discharge_ops.get('rx', 0) == 16

gcd_ops = build_gcd_hea(add_measurements=False).count_ops()
assert gcd_ops.get('ry', 0) == 20
assert gcd_ops.get('cz', 0) == 8

decoder_ops = build_eis_decoder_hea(add_measurements=False).count_ops()
assert decoder_ops.get('ry', 0) == 21
assert decoder_ops.get('cx', 0) == 6

qubo_ops = build_eis_qubo_qaoa(add_measurements=False).count_ops()
assert qubo_ops.get('h', 0) == 21
assert qubo_ops.get('rz', 0) == 21
assert qubo_ops.get('rzz', 0) == 20
assert qubo_ops.get('rx', 0) == 21

discharge_display_fig = draw_discharge_window_qaoa_display(show=False)
discharge_labels = figure_text_labels(discharge_display_fig)
assert len(discharge_display_fig.axes) == 1
assert 'c' in discharge_labels
assert any('\\gamma_' in label and 'a_i' in label and '1' in label for label in discharge_labels)
assert any('\\gamma_' in label and 'a_i' in label and '2' in label for label in discharge_labels)
assert any('\\mathrm{ZZ}' in label and '1' in label for label in discharge_labels)
assert any('\\mathrm{ZZ}' in label and '2' in label for label in discharge_labels)
assert any('U_{\\mathrm{len}}' in label and '1' in label for label in discharge_labels)
assert any('U_{\\mathrm{len}}' in label and '2' in label for label in discharge_labels)
assert any('\\beta_{1}' in label for label in discharge_labels)
assert any('\\beta_{2}' in label for label in discharge_labels)
assert '1.1' not in discharge_labels
assert '0.6' not in discharge_labels
plt.close(discharge_display_fig)

gcd_display_fig = draw_gcd_hea_display(show=False)
gcd_labels = figure_text_labels(gcd_display_fig)
assert 'c' in gcd_labels
assert any('\\theta_{1}' in label for label in gcd_labels)
assert any('\\phi_{1}' in label for label in gcd_labels)
assert any('\\theta_{2}' in label for label in gcd_labels)
assert any('\\phi_{2}' in label for label in gcd_labels)
plt.close(gcd_display_fig)

decoder_display_fig = draw_eis_decoder_hea_display(show=False)
decoder_labels = figure_text_labels(decoder_display_fig)
assert 'c' in decoder_labels
assert any('\\theta_{1,i}' in label for label in decoder_labels)
assert any('\\theta_{2,i}' in label for label in decoder_labels)
assert any('\\theta_{3,i}' in label for label in decoder_labels)
plt.close(decoder_display_fig)

compact_qubo_fig = draw_eis_qubo_qaoa_compact_view(show=False)
compact_labels = figure_text_labels(compact_qubo_fig)
assert len(compact_qubo_fig.axes) == 1
assert 'c' in compact_labels
assert any('\\gamma' in label and 'h_i' in label for label in compact_labels)
assert any('\\gamma' in label and 'J_{i,i+1}' in label for label in compact_labels)
assert any('\\beta' in label for label in compact_labels)
assert any('\\vdots' in label for label in compact_labels)
assert '0.048' not in compact_labels
assert '0.42' not in compact_labels
plt.close(compact_qubo_fig)

print('All notebook smoke checks passed.')
print('Discharge QAOA qubits:', build_discharge_window_qaoa().num_qubits)
print('GCD HEA qubits:', build_gcd_hea().num_qubits)
print('EIS decoder HEA qubits:', build_eis_decoder_hea().num_qubits)
print('EIS QUBO QAOA qubits:', build_eis_qubo_qaoa().num_qubits)
